# NLP Assignment - 5  
## Token Classification using Transformers  

### Objective  
This notebook demonstrates token classification using a transformer model (BERT).

### Tasks Covered  
- Dataset Loading  
- Tokenization  
- Label Alignment  
- Model Training  
- Evaluation  
- Inference  

### Tools Used  
- Python  
- Hugging Face Transformers  
- Datasets Library  
- PyTorch

## Note
Due to computational limitations, full training outputs may not be displayed.
Please run the notebook to reproduce results.

## Pipeline Flow
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference

In [ ]:
!pip install -q transformers datasets seqeval

In [ ]:
!pip install datasets==2.19.0

##Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)

##Label Names

In [ ]:
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)

##Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

##Tokenization + Label Alignment

In [ ]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(example["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

##Apply Tokenization

In [ ]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

##Load Model

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_names)
)

##Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=0.1,
    weight_decay=0.01
)

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

##Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

##Train

In [ ]:
trainer.train()

##Evaluate

In [ ]:
trainer.evaluate()

In [ ]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

text = "Barack Obama visited India"
print(nlp(text))